# Validación de las likelihoods

Para cada dataset (Pantheon, Pantheon+, Pantheon+ + SH0ES, CC, BAO full, DESI DR2, AGN):
1. **Datos:** número de puntos y comparación con la fuente original.
2. **$\chi^2$ del código contra una implementación independiente:** distancias con `astropy`, $r_d$ con `CAMB` y covarianzas leídas por otro camino. Diferencia en **%**.
3. **Best-fit ΛCDM contra resultados publicados** (cuando corresponde).

Requiere `classy` compilado desde CLASS vainilla (se usa para $r_d$ en BAO y DESI).

Todo en ΛCDM (sin radiación, como el código).

In [1]:
import os, sys, types, warnings, time
import numpy as np, pandas as pd
from scipy.optimize import minimize
from astropy.cosmology import FlatLambdaCDM
import camb
warnings.filterwarnings('ignore')

import git
path_git = git.Repo('.', search_parent_directories=True).working_tree_dir
sys.path.append(os.path.join(path_git, 'fr_mcmc', 'utils'))
try:
    import torch
except ImportError:  # chi_square importa ML (torch); la rama ML no se usa acá
    ml = types.ModuleType('ML'); ml.H_ML = None; ml.ML_limits = lambda m: (0, 0, 0, 0)
    sys.modules['ML'] = ml

import chi_square as cs
from data import (read_data_pantheon, read_data_pantheon_plus, read_data_pantheon_plus_shoes,
                  read_data_chronometers, read_data_BAO_full, read_data_DESI, read_data_AGN)
from BAO import r_drag, r_drag_class
import classy
print('classy:', classy.__file__, '(tiene que ser CLASS vainilla)')
from constants import WB_BBN

S = os.path.join(path_git, 'fr_mcmc', 'source')
c_km = 299792.458

def chi2_code(theta, **datasets):
    '''chi2 del código en LCDM, theta = [M, Om, H0].'''
    M, om, H0 = theta
    return cs.params_to_chi2([M, 147, om, H0], None, index=4, model='LCDM', **datasets)

def pct(a, b):
    return 100*abs(a/b - 1)

def fit(f, x0):
    r = minimize(f, x0, method='Nelder-Mead', options=dict(xatol=1e-7, fatol=1e-7, maxiter=40000))
    return r.x, r.fun

def cosmo(om, H0):
    return FlatLambdaCDM(H0=H0, Om0=om, Tcmb0=0)

results = []

classy: /tmp/claude-1000/-home-matias-Documents-PhD-code-fR-MCMC/39b720fc-b73c-41f1-bc54-2cf1b0443bee/scratchpad/class_vanilla/site/classy/__init__.py (tiene que ser CLASS vainilla)


## 1. Pantheon (Scolnic et al. 2018)

$\mu = 25 + 5\log_{10}[(1+z_{hel})\,c\int_0^{z_{cmb}}dz/H]$, $\chi^2=\Delta^T C^{-1}\Delta$ con $\Delta = m_B - M - \mu$ y $C = C_{sys} + \mathrm{diag}(\sigma_{m_B}^2)$.

In [2]:
os.chdir(os.path.join(S, 'Pantheon')); ds_SN = read_data_pantheon('lcparam_full_long_zhel.txt')
d = np.loadtxt('lcparam_full_long_zhel.txt', usecols=(1, 2, 4, 5))
Cp = np.loadtxt('lcparam_full_long_sys.txt').reshape(len(d), len(d)) + np.diag(d[:, 3]**2)
Cp_inv = np.linalg.inv(Cp)

def chi2_SN_ind(M, om, H0):
    mu = 5*np.log10((1 + d[:, 1])/(1 + d[:, 0])*cosmo(om, H0).luminosity_distance(d[:, 0]).value) + 25
    r = d[:, 2] - M - mu
    return r @ Cp_inv @ r

a, b = chi2_code([-19.3, 0.3, 70], dataset_SN=ds_SN), chi2_SN_ind(-19.3, 0.3, 70)
x, f = fit(lambda x: chi2_code([x[0], x[1], 70], dataset_SN=ds_SN), [-19.3, 0.3])
print(f'N = {len(d)}   chi2 código = {a:.4f}   independiente = {b:.4f}   diferencia = {pct(a, b):.1e} %')
print(f'Best-fit: Om = {x[1]:.4f}, chi2 = {f:.2f}      Scolnic+18 (SN solas): Om = 0.298 ± 0.022')
results.append(dict(dataset='Pantheon', N=len(d), **{'Δχ² código vs indep [%]': pct(a, b)}, best_fit=f'Om={x[1]:.3f}', referencia='Om=0.298±0.022'))

N = 1048   chi2 código = 1162.3667   independiente = 1162.3667   diferencia = 4.4e-06 %
Best-fit: Om = 0.2981, chi2 = 1026.86      Scolnic+18 (SN solas): Om = 0.298 ± 0.022


## 2. Pantheon+ sin SH0ES (Brout et al. 2022)

Se usan las SNe del flujo de Hubble ($z_{HD}>0.01$, $N=1590$) con la covarianza STAT+SYS. $M$ y $H_0$ son degenerados, así que se fija $H_0=70$.

Es la misma definición que las likelihoods oficiales de cobaya: `PantheonPlus` y `PantheonPlusShoes` leen el **mismo** `Pantheon+SH0ES.dat` y el **mismo** `Pantheon+SH0ES_STAT+SYS.cov` (es la única covarianza publicada). Solo difieren en:
- **Pantheon+:** máscara `zHD > 0.01`; $\mu$ cosmológico para todas las SNe.
- **Pantheon+ + SH0ES:** máscara `(zHD > 0.01) | IS_CALIBRATOR`; en los calibradores $\mu$ = `CEPH_DIST`.

**Correcciones aplicadas:**
- El $\chi^2$ no tenía rama para `dataset_SN_plus` y devolvía 0.
- La matriz `covmat_pantheon_plus_only.npz` usaba la máscara de PP+SH0ES ($z_{HD}>0.01$ **o** calibrador, $N=1657$). Sin distancias cefeidas, las 67 SNe calibradoras con $z<0.01$ entran como puntos de flujo de Hubble y sesgan $\Omega_m$ (abajo se muestra el efecto).

In [3]:
os.chdir(os.path.join(S, 'Pantheon_plus_shoes'))
t = time.time()
ds_PP = read_data_pantheon_plus('Pantheon+SH0ES.dat', 'Pantheon+SH0ES_STAT+SYS.cov')
ds_PPS = read_data_pantheon_plus_shoes('Pantheon+SH0ES.dat', 'Pantheon+SH0ES_STAT+SYS.cov')
print(f'lectura: {time.time() - t:.1f} s')

df = pd.read_csv('Pantheon+SH0ES.dat', sep=r'\s+')
C_full = np.loadtxt('Pantheon+SH0ES_STAT+SYS.cov', skiprows=1).reshape(len(df), len(df))

def chi2_pp_ind(M, om, H0, mask, cepheids):
    dd = df[mask]; Cinv = np.linalg.inv(C_full[np.ix_(mask, mask)])
    mu = 5*np.log10((1 + dd.zHEL.values)/(1 + dd.zHD.values)*cosmo(om, H0).luminosity_distance(dd.zHD.values).value) + 25
    if cepheids:
        mu = np.where(dd.IS_CALIBRATOR.values == 1, dd.CEPH_DIST.values, mu)
    r = dd.m_b_corr.values - M - mu
    return r @ Cinv @ r

m_PP = (df.zHD > 0.01).values
m_PPS = ((df.zHD > 0.01) | (df.IS_CALIBRATOR == 1)).values

a, b = chi2_code([-19.3, 0.3, 70], dataset_SN_plus=ds_PP), chi2_pp_ind(-19.3, 0.3, 70, m_PP, False)
x, f = fit(lambda x: chi2_code([x[0], x[1], 70], dataset_SN_plus=ds_PP), [-19.3, 0.3])
print(f'N = {m_PP.sum()}   chi2 código = {a:.4f}   independiente = {b:.4f}   diferencia = {pct(a, b):.1e} %')
print(f'Best-fit: Om = {x[1]:.4f}, chi2 = {f:.2f}      Brout+22 (SN solas): Om = 0.334 ± 0.018')
results.append(dict(dataset='Pantheon+', N=int(m_PP.sum()), **{'Δχ² código vs indep [%]': pct(a, b)}, best_fit=f'Om={x[1]:.3f}', referencia='Om=0.334±0.018'))

x_old, _ = fit(lambda x: chi2_pp_ind(x[0], x[1], 70, m_PPS, False), [-19.3, 0.3])
print(f'Con la máscara anterior (N={m_PPS.sum()}, calibradores sin cefeidas): Om = {x_old[1]:.4f}  <- sesgado')

lectura: 16.4 s


N = 1590   chi2 código = 1673.7845   independiente = 1673.7846   diferencia = 6.4e-06 %
Best-fit: Om = 0.3316, chi2 = 1402.92      Brout+22 (SN solas): Om = 0.334 ± 0.018


Con la máscara anterior (N=1657, calibradores sin cefeidas): Om = 0.3606  <- sesgado


## 3. Pantheon+ + SH0ES

Para las SNe en galaxias con cefeidas (`IS_CALIBRATOR=1`), $\mu$ es la distancia cefeida (`CEPH_DIST`); para el resto, la cosmológica. Se usa $z_{HD}>0.01$ o calibrador ($N=1657$).

In [4]:
a, b = chi2_code([-19.25, 0.3, 73], dataset_SN_plus_shoes=ds_PPS), chi2_pp_ind(-19.25, 0.3, 73, m_PPS, True)
x, f = fit(lambda x: chi2_code(x, dataset_SN_plus_shoes=ds_PPS), [-19.25, 0.3, 73])
print(f'N = {m_PPS.sum()}   chi2 código = {a:.4f}   independiente = {b:.4f}   diferencia = {pct(a, b):.1e} %')
print(f'Best-fit: M = {x[0]:.3f}, Om = {x[1]:.4f}, H0 = {x[2]:.2f}, chi2 = {f:.2f}')
print('Brout+22 (PP+SH0ES, ΛCDM plano): H0 = 73.5 ± 1.1, Om = 0.334 ± 0.018')
results.append(dict(dataset='Pantheon+ + SH0ES', N=int(m_PPS.sum()), **{'Δχ² código vs indep [%]': pct(a, b)}, best_fit=f'H0={x[2]:.2f}, Om={x[1]:.3f}', referencia='H0=73.5±1.1, Om=0.334±0.018'))

N = 1657   chi2 código = 1484.0922   independiente = 1484.0923   diferencia = 1.7e-06 %
Best-fit: M = -19.244, Om = 0.3318, H0 = 73.53, chi2 = 1452.02
Brout+22 (PP+SH0ES, ΛCDM plano): H0 = 73.5 ± 1.1, Om = 0.334 ± 0.018


## 4. Cronómetros cósmicos (CC)

$\chi^2 = \sum_i (H_i - H(z_i))^2/\sigma_i^2$ (diagonal).

In [5]:
os.chdir(os.path.join(S, 'CC')); ds_CC = read_data_chronometers('chronometers_data.txt')
z, H, e = ds_CC
chi2_CC_ind = lambda om, H0: np.sum(((cosmo(om, H0).H(z).value - H)/e)**2)
a, b = chi2_code([0, 0.3, 70], dataset_CC=ds_CC), chi2_CC_ind(0.3, 70)
x, f = fit(lambda x: chi2_code([0, x[0], x[1]], dataset_CC=ds_CC), [0.3, 70])
print(f'N = {len(z)}   chi2 código = {a:.6f}   independiente = {b:.6f}   diferencia = {pct(a, b):.1e} %')
print(f'Best-fit: Om = {x[0]:.4f}, H0 = {x[1]:.2f}, chi2 = {f:.2f}')
results.append(dict(dataset='CC', N=len(z), **{'Δχ² código vs indep [%]': pct(a, b)}, best_fit=f'H0={x[1]:.1f}, Om={x[0]:.3f}', referencia='-'))

N = 30   chi2 código = 14.979087   independiente = 14.979087   diferencia = 2.9e-08 %
Best-fit: Om = 0.3196, H0 = 68.14, chi2 = 14.49


## 5. Horizonte de sonido $r_d$

BAO full y DESI usan $r_d(\Omega_m, H_0, \omega_b^{BBN}=0.02218)$.
- **Antes:** integral de $c_s/H$ desde el $z_d$ de Eisenstein & Hu con $R = \omega_b/\omega_\gamma$. Le falta el factor 3/4 ($R = 3\rho_b/4\rho_\gamma$) y el $z_d$ del fit no es el de CAMB. Los dos errores se compensan en parte y el resultado queda un ~0.85 % por debajo.
- **Ahora:** CLASS vainilla en cada llamada (`r_drag_class`, ~70 ms), con $\sum m_\nu = 0.06$ eV y $N_{eff}=3.044$.

Como referencia independiente se usa CAMB.

In [6]:
def rd_camb(om, H0, wb=WB_BBN):
    h = H0/100
    p = camb.set_params(H0=H0, ombh2=wb, omch2=om*h*h - wb - 0.06/93.14, mnu=0.06, nnu=3.044)
    return camb.get_background(p).get_derived_params()['rdrag']

rows = []
for om in [0.25, 0.3, 0.35]:
    for H0 in [65, 70, 75]:
        r = rd_camb(om, H0)
        t = time.time(); rc = r_drag_class(om, H0, WB_BBN); dt = time.time() - t
        rows.append(dict(Om=om, H0=H0, rd_CAMB=r, rd_CLASS=rc, **{'anterior vs CAMB [%]': 100*(r_drag(om, H0, WB_BBN)/r - 1),
                                                   'CLASS vs CAMB [%]': 100*(rc/r - 1)}, t_CLASS_ms=1e3*dt))
pd.DataFrame(rows)

,Om,H0,rd_CAMB,rd_CLASS,anterior vs CAMB [%],CLASS vs CAMB [%],t_CLASS_ms
0,0.25,65,158.486754,158.505654,-0.766154,0.011925,96.804619
1,0.25,70,153.060439,153.072801,-0.829599,0.008077,72.110653
2,0.25,75,147.898869,147.904621,-0.868073,0.003889,72.569847
3,0.30,65,151.793486,151.804286,-0.841269,0.007115,70.553541
4,0.30,70,146.222238,146.225763,-0.874930,0.002411,70.877314
5,0.30,75,140.960677,140.956806,-0.875311,-0.002747,70.670843
6,0.35,65,145.997226,146.000446,-0.875617,0.002206,70.481300
7,0.35,70,140.340778,140.336034,-0.873029,-0.003380,85.259676
8,0.35,75,135.029616,135.016765,-0.830955,-0.009517,72.732687


## 6. DESI DR2 (DESI Collaboration 2025, arXiv:2503.14738)

Los datos del repositorio eran de **DR1**. Se agregaron los archivos oficiales de DR2 (`source/DESI/DR2_official/`, de CobayaSampler/bao_data) y, a partir de ellos, `DESI_DR2_dm_dh.txt` y `DESI_DR2_dv.txt`. En DR2 el bin QSO mide $D_M/r_d$ y $D_H/r_d$ (en DR1 era $D_V/r_d$).

In [7]:
os.chdir(os.path.join(S, 'DESI'))
ds_DESI = read_data_DESI('DESI_DR2_dm_dh.txt', 'DESI_DR2_dv.txt')
mean = [l.split() for l in open('DR2_official/desi_gaussian_bao_ALL_GCcomb_mean.txt') if not l.startswith('#')]
z_off = np.array([float(m[0]) for m in mean]); v_off = np.array([float(m[1]) for m in mean]); q_off = [m[2] for m in mean]
C_off = np.loadtxt('DR2_official/desi_gaussian_bao_ALL_GCcomb_cov.txt'); Cinv_off = np.linalg.inv(C_off)

# 1) Archivos del código vs oficiales: se reconstruye la covarianza a partir de (σ, ρ)
(z1, dm, edm, dh, edh, rho), (z2, dv, edv) = ds_DESI
C_rec = np.zeros_like(C_off); v_rec = np.zeros_like(v_off)
for i, (zz, qq) in enumerate(zip(z_off, q_off)):
    if qq == 'DV_over_rs':
        k = np.argmin(abs(z2 - zz)); v_rec[i] = dv[k]; C_rec[i, i] = edv[k]**2
    else:
        k = np.argmin(abs(z1 - zz))
        v_rec[i] = dm[k] if qq == 'DM_over_rs' else dh[k]
        for j, (zz2, qq2) in enumerate(zip(z_off, q_off)):
            if zz2 == zz and qq2 != 'DV_over_rs':
                si = edm[k] if qq == 'DM_over_rs' else edh[k]; sj = edm[k] if qq2 == 'DM_over_rs' else edh[k]
                C_rec[i, j] = si*sj*(1 if qq == qq2 else rho[k])
print(f'max |Δ valores| = {100*np.max(abs(v_rec/v_off - 1)):.1e} %,  max |Δ covarianza| = {100*np.max(abs(C_rec - C_off))/np.max(abs(C_off)):.1e} %')

# 2) chi2 del código vs independiente (covarianza oficial, astropy, CAMB)
def chi2_DESI_ind(om, H0):
    cc = cosmo(om, H0); rd = rd_camb(om, H0)
    th = []
    for zz, qq in zip(z_off, q_off):
        DM = cc.comoving_transverse_distance(zz).value; DH = c_km/cc.H(zz).value
        th.append({'DM_over_rs': DM, 'DH_over_rs': DH, 'DV_over_rs': (zz*DM**2*DH)**(1/3)}[qq]/rd)
    r = np.array(th) - v_off
    return r @ Cinv_off @ r

a, b = chi2_code([0, 0.3, 68], dataset_DESI=ds_DESI), chi2_DESI_ind(0.3, 68)
x, f = fit(lambda x: chi2_code([0, x[0], x[1]], dataset_DESI=ds_DESI), [0.3, 68])
print(f'N = {len(v_off)}   chi2 código = {a:.4f}   independiente = {b:.4f}   diferencia = {pct(a, b):.1e} %')
print(f'Best-fit (ω_b = {WB_BBN} fijo): Om = {x[0]:.4f}, H0 = {x[1]:.2f}, chi2 = {f:.2f}')
print('DESI DR2 (BAO solo): Om = 0.2975 ± 0.0086')
results.append(dict(dataset='DESI DR2', N=len(v_off), **{'Δχ² código vs indep [%]': pct(a, b)}, best_fit=f'Om={x[0]:.4f}, H0={x[1]:.2f}', referencia='Om=0.2975±0.0086'))

max |Δ valores| = 5.6e-08 %,  max |Δ covarianza| = 6.1e-06 %


N = 13   chi2 código = 12.5619   independiente = 12.6088   diferencia = 3.7e-01 %
Best-fit (ω_b = 0.02218 fijo): Om = 0.2975, H0 = 68.54, chi2 = 10.27
DESI DR2 (BAO solo): Om = 0.2975 ± 0.0086


## 7. BAO full

Compilación propia:
- MGS ($z=0.15$) y WiggleZ ($z=0.44$): $D_V/r_d$.
- BOSS DR12 consenso ($z=0.38, 0.51$): $D_M/r_d$ y $H\,r_d$, obtenidos de Alam et al. 2017 con $r_{d,fid}=147.78$ Mpc.
- DESI **DR1**: QSO $D_V/r_d$, y LRG2, LRG3+ELG1, ELG2 con $D_M/r_d$, $D_H/r_d$ y $\rho$.
- Lyα: $D_M/r_d$ y $D_H/r_d$.

In [8]:
os.chdir(os.path.join(S, 'BAO_legacy_2')); ds_BAOf = read_data_BAO_full('BAO_full_1.csv', 'BAO_full_2.csv')
d1 = pd.read_csv('BAO_full_1.csv'); d2 = pd.read_csv('BAO_full_2.csv')
print('Conversión BOSS DR12 (Alam+17): D_M(0.38)=1518, D_M(0.51)=1977 Mpc; H(0.38)=81.5, H(0.51)=90.5 km/s/Mpc; r_d,fid=147.78')
print('  D_M/r_d:', np.round([1518/147.78, 1977/147.78], 3), ' archivo:', d1.Dist[d1['index'] == 2].values)
print('  H r_d  :', np.round([81.5*147.78, 90.5*147.78], 2), ' archivo:', d1.Dist[d1['index'] == 4].values)

def chi2_BAOf_ind(om, H0):
    cc = cosmo(om, H0); rd = rd_camb(om, H0); chi2 = 0
    for _, r in d1.iterrows():
        DM = cc.comoving_transverse_distance(r.z).value; DH = c_km/cc.H(r.z).value
        th = {3: (r.z*DM**2*DH)**(1/3)/rd, 2: DM/rd, 4: cc.H(r.z).value*rd}[int(r['index'])]
        chi2 += (th - r.Dist)**2/(r.Stat_error**2 + r.Sist_error**2)
    for _, r in d2.iterrows():
        DM = cc.comoving_transverse_distance(r.z_eff).value/rd; DH = c_km/cc.H(r.z_eff).value/rd
        Cm = np.array([[r.error_Dh_rd**2, r.rho*r.error_Dh_rd*r.error_Dm_rd], [r.rho*r.error_Dh_rd*r.error_Dm_rd, r.error_Dm_rd**2]])
        dlt = np.array([DH - r.Dh_rd, DM - r.Dm_rd]); chi2 += dlt @ np.linalg.solve(Cm, dlt)
    return chi2

a, b = chi2_code([0, 0.3, 68], dataset_BAO_full=ds_BAOf), chi2_BAOf_ind(0.3, 68)
x, f = fit(lambda x: chi2_code([0, x[0], x[1]], dataset_BAO_full=ds_BAOf), [0.3, 68])
print(f'N = {len(d1) + 2*len(d2)}   chi2 código = {a:.4f}   independiente = {b:.4f}   diferencia = {pct(a, b):.1e} %')
print(f'Best-fit: Om = {x[0]:.4f}, H0 = {x[1]:.2f}, chi2 = {f:.2f}')
results.append(dict(dataset='BAO full', N=len(d1) + 2*len(d2), **{'Δχ² código vs indep [%]': pct(a, b)}, best_fit=f'Om={x[0]:.3f}, H0={x[1]:.2f}', referencia='-'))

Conversión BOSS DR12 (Alam+17): D_M(0.38)=1518, D_M(0.51)=1977 Mpc; H(0.38)=81.5, H(0.51)=90.5 km/s/Mpc; r_d,fid=147.78
  D_M/r_d: [10.272 13.378]  archivo: [10.272 13.378]
  H r_d  : [12044.07 13374.09]  archivo: [12044.07 13374.09]


N = 15   chi2 código = 10.1409   independiente = 10.1415   diferencia = 5.3e-03 %
Best-fit: Om = 0.2984, H0 = 68.00, chi2 = 10.12


## 8. AGN / cuásares (Lusso et al. 2020, $N=2421$)

$\log_{10}(d_LH_0)_{obs} = \log_{10}(3.24)-25+\dfrac{\log F_X-\gamma\log F_{UV}-\beta}{2\gamma-2}$ con $\beta=7.735\pm0.244$ y $\gamma=0.648\pm0.007$ fijos (Li et al. 2021), y

$\sigma^2 = \dfrac{\sigma_{F_X}^2+\gamma^2\sigma_{F_{UV}}^2+\sigma_\beta^2}{(2\gamma-2)^2} + \left(\dfrac{\partial}{\partial\gamma}\right)^2\sigma_\gamma^2$.

In [9]:
os.chdir(os.path.join(S, 'AGN')); ds_AGN = read_data_AGN('table3.dat')
zq, lFuv, eFuv, lFx, eFx = ds_AGN
beta, ebeta, gamma, egamma = 7.735, 0.244, 0.648, 0.007

def obs_logdlH0(beta, gamma):
    return np.log10(3.24) - 25 + (lFx - gamma*lFuv - beta)/(2*gamma - 2)

def chi2_AGN_ind(om):
    th = np.log10(cosmo(om, 70).luminosity_distance(zq).value*70)
    dg = (obs_logdlH0(beta, gamma + 1e-6) - obs_logdlH0(beta, gamma - 1e-6))/2e-6   # derivada numérica en γ
    var = (eFx**2 + gamma**2*eFuv**2 + ebeta**2)/(2*gamma - 2)**2 + dg**2*egamma**2
    return np.sum((th - obs_logdlH0(beta, gamma))**2/var), var

a = chi2_code([0, 0.3, 70], dataset_AGN=ds_AGN); b, var = chi2_AGN_ind(0.3)
print(f'N = {len(zq)}   chi2 código = {a:.4f}   independiente = {b:.4f}   diferencia = {pct(a, b):.1e} %')
print(f'Independencia de H0: chi2(H0=60) - chi2(H0=80) = {chi2_code([0, .3, 60], dataset_AGN=ds_AGN) - chi2_code([0, .3, 80], dataset_AGN=ds_AGN):.1e}')
x, f = fit(lambda x: chi2_code([0, x[0], 70], dataset_AGN=ds_AGN), [0.3])
print(f'Best-fit: Om = {x[0]:.3f}, chi2 = {f:.1f}, chi2/N = {f/len(zq):.3f}')
tot = np.median(var)
print('Presupuesto de error (mediana de σ² en dex², log d_L H0):')
print(f'  flujos      {np.median((eFx**2 + gamma**2*eFuv**2)/(2*gamma - 2)**2):.3f}')
print(f'  σ_β         {ebeta**2/(2*gamma - 2)**2:.3f}')
print(f'  σ_γ         {np.median(var - (eFx**2 + gamma**2*eFuv**2 + ebeta**2)/(2*gamma - 2)**2):.3f}')
print(f'  total       {tot:.3f}  ->  σ ≈ {np.sqrt(tot):.2f} dex')
results.append(dict(dataset='AGN', N=len(zq), **{'Δχ² código vs indep [%]': pct(a, b)}, best_fit=f'Om={x[0]:.3f}, χ²/N={f/len(zq):.2f}', referencia='ver texto'))

N = 2421   chi2 código = 1169.1264   independiente = 1169.1264   diferencia = 8.7e-09 %
Independencia de H0: chi2(H0=60) - chi2(H0=80) = -3.4e-12


Best-fit: Om = 0.395, chi2 = 1161.6, chi2/N = 0.480
Presupuesto de error (mediana de σ² en dex², log d_L H0):
  flujos      0.004
  σ_β         0.120
  σ_γ         0.108
  total       0.233  ->  σ ≈ 0.48 dex


**Observaciones sobre AGN** (la implementación reproduce la fórmula, pero el modelo estadístico es discutible):
1. $\sigma_\beta$ y $\sigma_\gamma$ son incertezas de parámetros **globales**: están 100 % correlacionadas entre los 2421 cuásares, pero se suman a cada punto como si fueran independientes. Eso infla los errores (dominan el presupuesto) y da $\chi^2/N\approx0.5$.
2. No hay dispersión intrínseca $\delta$ de la relación $L_X$–$L_{UV}$ (~0.23 dex en Lusso & Risaliti), salvo que se la considere absorbida en $\sigma_\beta$.
3. Alternativa estándar: dejar $\beta$, $\gamma$ (y $\delta$) como parámetros nuisance con priors gaussianos de la calibración, o usar la covarianza completa $C = \mathrm{diag}(\sigma_i^2) + J_\beta J_\beta^T\sigma_\beta^2 + J_\gamma J_\gamma^T\sigma_\gamma^2$.

## Resumen

In [10]:
pd.set_option('display.float_format', lambda x: '%.2g' % x)
pd.DataFrame(results)

,dataset,N,Δχ² código vs indep [%],best_fit,referencia
0,Pantheon,1048,4.4e-06,Om=0.298,Om=0.298±0.022
1,Pantheon+,1590,6.4e-06,Om=0.332,Om=0.334±0.018
2,Pantheon+ + SH0ES,1657,1.7e-06,"H0=73.53, Om=0.332","H0=73.5±1.1, Om=0.334±0.018"
3,CC,30,2.9e-08,"H0=68.1, Om=0.320",-
4,DESI DR2,13,0.37,"Om=0.2975, H0=68.54",Om=0.2975±0.0086
5,BAO full,15,0.0053,"Om=0.298, H0=68.00",-
6,AGN,2421,8.7e-09,"Om=0.395, χ²/N=0.48",ver texto
